# ðŸš€ MyUPI Voice + Action Assistant on Google Colab (GPU)

This notebook runs the complete **MyUPI Voice Assistant** on a free Google Colab GPU with full **Voice-In (ASR)** and **Voice-Out (TTS)** enabled!

### Step 1: Check GPU
Ensure you are connected to a GPU runtime (`Runtime > Change runtime type > T4 GPU`).

In [ ]:
!nvidia-smi

### Step 2: Install System and Python Dependencies
(Works universally on Python 3.10, 3.11, 3.12, and 3.13)

In [ ]:
!apt-get install -y ffmpeg
!pip install -r requirements.txt
!pip install git+https://github.com/huggingface/parler-tts.git
!pip install faster-whisper edge-tts pycloudflared


### Step 3: Login to Hugging Face
The Bodhan AI ASR model requires you to accept the terms on Hugging Face (https://huggingface.co/bodhan-ai/indic-transcribe-flex). Generate a token at huggingface.co/settings/tokens and paste it here.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

### Step 4: Configure `config/config.yaml` for Indic Voice Models

In [ ]:
import yaml

with open("config/config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

# Set ASR to Bodhan AI Indic-Transcribe-Flex
cfg["asr"]["provider"] = "indic_transcribe"
cfg["asr"]["model_authoritative"] = "bodhan-ai/indic-transcribe-flex"
cfg["asr"]["device"] = "cuda"

# Set TTS to AI4Bharat Indic-Parler-TTS
cfg["tts"]["provider"] = "indic_parler"
cfg["tts"]["model"] = "ai4bharat/indic-parler-tts"

with open("config/config.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("✅ config.yaml updated for Indic Voice Pipeline!")

### Step 5: Launch Server with Public HTTPS URL (Cloudflare Tunnel)
Click the generated `trycloudflare.com` public URL to open the MyUPI Web App in your browser and test voice!

In [ ]:
from pycloudflared import try_cloudflare

# Start Cloudflare tunnel on port 8000
tunnel_url = try_cloudflare(port=8000)
print(f"\nðŸŽ‰ YOUR PUBLIC APP URL: {tunnel_url.tunnel}\n")
print("Open this link on your phone or laptop to speak directly with the AI!\n")

# Launch FastAPI server
!uvicorn app.main:app --host 0.0.0.0 --port 8000